# Agente de compras con memoria

Qwen2.5-7B-Instruct coordina nueve herramientas. El código valida cantidades, presupuesto, consentimiento y stock; la memoria conserva preferencias, carrito, cotizaciones y pedidos.

In [8]:
%pip install "transformers==4.46.2"
%pip install "bitsandbytes>=0.43"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [9]:
import bitsandbytes as bnb
import transformers
import accelerate

print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

bitsandbytes: 0.50.2
transformers: 4.46.2
accelerate: 1.16.0


In [10]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from IPython.display import display
from lab4_asistente.runtime import cargar_sistema


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Cargar los componentes

In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
print("Dispositivo:", COMP["dispositivo"])
print("Carga inicial por componente:", COMP["cargas"])
print("Herramientas disponibles:", len(__import__("lab4_asistente.herramientas", fromlist=["ESQUEMAS"]).ESQUEMAS))


La VRAM disponible no alcanza para el modelo cuantizado; se cargará en CPU.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Interacción por turnos

In [ ]:
SIS.reiniciar()
r1 = SIS.agente.turno("Busco una mochila para portátil negra, máximo 40 euros")
print(r1["respuesta"])
print("Herramientas:", [(o["herramienta"], o["resultado"].get("ok")) for o in r1["operaciones"]])
r2 = SIS.agente.turno("¿Qué opciones encontraste?")
print(r2["respuesta"])
print("Estado persistente:")
print(SIS.h.resumen_estado())


Encontré varias opciones para usted. ¿Le interesa la mochila Besttravel por 31.99 euros o la FUXINGYAO también por 31.99 euros? Ambas son negras y están disponibles.
Herramientas: [('buscar_productos', True)]
He encontrado la mochila Besttravel y la FUXINGYO, ambas negras y por 31.99 euros. ¿Cuál prefiere?
Estado persistente:
ESTADO ACTUAL (memoria del sistema):
- Presupuesto total maximo: no indicado euros
- Preferencias: ninguna
- Carrito: vacio
- Productos mostrados en la ultima busqueda:
  1) B07Q5S8K28 | Mochila Unisex Impermeable para Ordenador Portátil de hasta 15.6 Pulgadas, con Puerto USB, | 31.99 euros | stock 9
  2) B093NYPDX5 | FUXINGYAO Mochila Unisex para Ordenador Portátil de hasta 15.6 PulgadasMochila Portatil Mo | 31.99 euros | stock 9
  3) B075Z16J1Z | Mochila de Portátil, REYLEO Mochila Hombre con Puerto de USB para Ordenador hasta 15,6 Pul | 10.99 euros | stock 0
  4) B074KV9TT4 | Mochila de viaje de negocios, mochila para computadora portátil con puerto USB para ho

## Verificación del camino crítico de compra

In [ ]:
# El agente debe presentar una cotización antes de poder registrar un pedido.
# La confirmación explícita se ejecuta y valida en código; la frase del LLM nunca basta.
print("Estado de cotización:", SIS.h.estado.cotizacion)
print("¿Cotización mostrada?:", SIS.h.estado.cotizacion_mostrada)


Estado de cotización: None
¿Cotización mostrada?: False


## Formato y reglas de compra

1. El cliente describe el producto y la cantidad. Si falta un dato necesario, el agente pregunta antes de modificar el carrito.
2. El agente busca productos y consulta precio y stock en las herramientas. Solo ofrece datos devueltos por catálogo y tienda.
3. A petición del cliente, agrega al carrito la cantidad total deseada. El carrito es una preparación; todavía no existe un pedido.
4. El agente genera una cotización o proforma con productos, cantidades, total y disponibilidad. La muestra y pide confirmación explícita.
5. Solo después de esa confirmación, la tienda vuelve a validar cotización, carrito, stock y presupuesto. Si todo sigue vigente, registra el pedido y descuenta inventario en una transacción.
6. Si el cliente cambia el carrito, cambian las condiciones o rechaza la proforma, no se registra el pedido. Se prepara y presenta una nueva cotización cuando corresponde. Una confirmación repetida no duplica la compra.

El agente conversa y elige herramientas; las herramientas validan reglas; SQLite conserva el estado real de carritos, cotizaciones, inventario y pedidos.

## Componentes y datos que intercambian

```mermaid
flowchart LR
    Cliente[Cliente] <--> Agente[Agente conversacional]
    Agente <--> Memoria[Memoria de sesión]
    Agente --> Herramientas[Herramientas validadas]
    Herramientas --> Catalogo[Catálogo y recuperación RAG]
    Herramientas <--> SQLite[(SQLite: carrito, proforma, stock y pedidos)]
    Herramientas --> Agente
    Agente --> Cliente
```

La respuesta al cliente puede ser una aclaración, resultados de búsqueda, una proforma pendiente o la confirmación de un pedido registrado. SQLite es la fuente de verdad para inventario y compras.

## Caso 1 — Compra confirmada

El cliente pide una mochila para portátil y especifica una unidad. El agente busca opciones disponibles, agrega la elegida al carrito, presenta la proforma y espera la confirmación. Solo después registra el pedido.

In [ ]:
SIS.reiniciar()
consulta_1 = "Busca una mochila para portátil negra de menos de 40 euros y agrega una unidad al carrito."
turno_1 = SIS.agente.turno(consulta_1)
print("Cliente:", consulta_1)
print("Agente:", turno_1["respuesta"])
print("Herramientas:", [(o["herramienta"], o["resultado"].get("ok")) for o in turno_1["operaciones"]])
print("Carrito tras la solicitud:")
display(pd.DataFrame(SIS.h._carrito()["items"]))
print("Cotización en memoria:", SIS.h.estado.cotizacion)
if SIS.h.estado.carrito_id:
    print("Detalle persistido del carrito:")
    display(pd.DataFrame(SIS.tienda.carrito(SIS.h.estado.carrito_id)["items"]))
print("Pedidos registrados:", SIS.h.estado.pedidos)

Cliente: Busca una mochila para portátil negra de menos de 40 euros y agrega una unidad al carrito.
Agente: He encontrado una mochila para portátil negra por 31.99 euros. ¿Desea agregarla al carrito?
Herramientas: [('buscar_productos', True)]
Carrito tras la solicitud:


NameError: name 'pd' is not defined

Ahora el cliente confirma la proforma que el agente acaba de mostrar. El agente ejecuta la confirmación y la tienda vuelve a validar precio, stock y presupuesto antes de registrar el pedido.

In [ ]:
confirmacion_1 = "Sí, confirmo la cotización."
turno_2 = SIS.agente.turno(confirmacion_1)
print("Cliente:", confirmacion_1)
print("Herramienta ejecutada:", [(o["herramienta"], o["resultado"]) for o in turno_2["operaciones"]])
print("Agente:", turno_2["respuesta"])
print("Estado SQLite:")
print("Pedidos:", SIS.tienda.resumen()["pedidos"])
print("Stock modificado:", SIS.tienda.resumen()["productos_modificados"])
assert any(o["herramienta"] == "confirmar_pedido" and o["resultado"].get("ok") for o in turno_2["operaciones"]), "No se registró la confirmación"

## Caso 2 — Aclaración antes de tocar el carrito

Una petición incompleta no debe crear líneas en el carrito. El agente pide el dato que falta y mantiene el carrito vacío.

In [ ]:
SIS.reiniciar()
consulta_2 = "Quiero comprar una mochila."
turno_3 = SIS.agente.turno(consulta_2)
print("Cliente:", consulta_2)
print("Agente:", turno_3["respuesta"])
print("Herramientas:", [(o["herramienta"], o["resultado"].get("ok")) for o in turno_3["operaciones"]])
print("Carrito:", SIS.h._carrito())
assert not SIS.h._carrito()["items"], "El carrito cambió antes de aclarar la petición"

## Caso 3 — Proforma pendiente de confirmación

Después de presentar una proforma, el cliente responde que lo pensará. Esa respuesta no autoriza la compra: el pedido debe seguir sin registrarse y el carrito queda disponible para continuar.

In [ ]:
SIS.reiniciar()
turno_4 = SIS.agente.turno("Busca una botella térmica y agrega una unidad al carrito.")
print("Agente presenta la cotización:", turno_4["respuesta"])
print("Cotización mostrada:", SIS.h.estado.cotizacion_mostrada)
pendiente = SIS.agente.turno("Déjame pensarlo.")
print("Cliente: Déjame pensarlo.")
print("Agente:", pendiente["respuesta"])
print("Operaciones en este turno:", [(o["herramienta"], o["resultado"].get("ok")) for o in pendiente["operaciones"]])
print("Cotización sigue vigente:", SIS.h.estado.cotizacion)
print("Carrito conservado:", SIS.h._carrito())
print("Pedidos registrados:", SIS.h.estado.pedidos)
assert not SIS.h.estado.pedidos, "Se registró un pedido sin confirmación explícita"

## Caso 4 — Rechazo explícito de la proforma

Si el cliente rechaza la compra, el agente conserva el estado sin pedido. Una negativa nunca se interpreta como confirmación, aunque exista una cotización vigente.

In [ ]:
rechazo_1 = SIS.agente.turno("No, cancela la compra.")
print("Cliente: No, cancela la compra.")
print("Agente:", rechazo_1["respuesta"])
print("Herramientas ejecutadas:", [(o["herramienta"], o["resultado"].get("ok")) for o in rechazo_1["operaciones"]])
print("Pedidos registrados:", SIS.h.estado.pedidos)
assert not SIS.h.estado.pedidos, "Una respuesta negativa no debe registrar la compra"

## Caso 5 — Cotización vencida por cambio en el inventario

La proforma representa el precio y el stock revisados al cotizar. Si el inventario cambia antes de la confirmación, la tienda rechaza el pedido de forma atómica. El agente informa el problema y debe preparar una nueva proforma.

In [ ]:
SIS.reiniciar()
turno_5 = SIS.agente.turno("Busca audífonos bluetooth y agrega una unidad al carrito.")
print("Proforma presentada:", turno_5["respuesta"])
items_antes = SIS.tienda.carrito(SIS.h.estado.carrito_id)["items"]
if items_antes:
    item_prueba = items_antes[0]
    SIS.tienda.fijar_precio(item_prueba["product_id"], item_prueba["precio_cent"] + 500)
    vencida = SIS.agente.turno("Sí, confirmo la cotización.")
    print("Resultado real de la confirmación:", [(o["herramienta"], o["resultado"]) for o in vencida["operaciones"]])
    print("Agente:", vencida["respuesta"])
    print("Pedidos registrados:", SIS.h.estado.pedidos)
    assert not SIS.h.estado.pedidos, "No debe registrarse una cotización con precio cambiado"
else:
    print("No hubo productos en el carrito; revisa la búsqueda para ejecutar este caso.")
    assert False, "No se pudo preparar el caso de cotización vencida"

## Caso 6 — Confirmación repetida sin duplicar

La base de datos asocia cada pedido a una cotización única. Repetir el mismo sí devuelve el pedido existente y no crea otra fila.

In [ ]:
SIS.reiniciar()
turno_6 = SIS.agente.turno("Busca un cargador USB y agrega una unidad al carrito.")
print("Proforma:", turno_6["respuesta"])
primera = SIS.agente.turno("Sí, confirmo.")
segunda = SIS.agente.turno("Sí, confirmo.")
print("Primera confirmación:", primera["respuesta"], [(o["resultado"].get("estado"), o["resultado"].get("pedido_id")) for o in primera["operaciones"]])
print("Confirmación repetida:", segunda["respuesta"], [(o["resultado"].get("estado"), o["resultado"].get("pedido_id")) for o in segunda["operaciones"]])
print("Pedidos en memoria:", SIS.h.estado.pedidos)
assert len(SIS.h.estado.pedidos) == 1, "La cotización produjo más de un pedido"
SIS.reiniciar()  # limpia tienda y memoria para continuar.